In [65]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re

# 5. Write Weight File
This notebook...

In [66]:
# Read in csv files
GT_df = pd.read_csv('../data/weight_files/GT_weights.csv')

LG_df = pd.read_csv('../data/weight_files/LG_weights.csv')
TRF_df = pd.read_csv('../data/weight_files/TRF_weights.csv')

spotlight_df = pd.read_csv('../data/weight_files/spotlight_weights.csv')

relik_df = pd.read_csv('../data/weight_files/relik_weights.csv')
relik_cie_df = pd.read_csv('../data/weight_files/relik_cie_weights.csv')

In [67]:
def extract_name(name):
    name = re.sub(r'_[a-z]{2}$', '', name)  
    name = name.replace('_', ' ') 
    name = name.replace("'", "") 
    return name.replace(',', '')                

In [68]:
relik_df['name'] = relik_df['name'].apply(extract_name)
relik_cie_df['name'] = relik_cie_df['name'].apply(extract_name)

## Map names to IDs

In [69]:
merged = GT_df.merge(LG_df, on='name').merge(TRF_df, on='name').merge(relik_df, on='name').merge(relik_cie_df, on='name').merge(spotlight_df, left_on='ID', right_on='source_wikidata_id')

In [70]:
merged = merged.drop(['name', 'source_wikidata_id', 'target_wikidata_id', 'language_code'], axis=1)
merged = merged.rename(columns={"ID": "source_ID", "weight": "spotlight_weight"})
merged.insert(1, 'target_ID', 'Q517')

In [71]:
merged['direction'] = 'in'

In [72]:
merged.head()

,source_ID,target_ID,GT_weight,LG_weight,TRF_weight,relik_weight,relik_cie_weight,spotlight_weight,direction
0,Q9068,Q517,1,1,1,0,3,1,in
1,Q360481,Q517,13,6,7,6,13,7,in
2,Q162296,Q517,2,2,2,2,5,2,in
3,Q354477,Q517,0,0,0,0,0,2,in
4,Q68771,Q517,0,0,0,0,16,3,in


In [73]:
merged.shape

(46, 9)

In [74]:
merged.isna().sum()

source_ID           0
target_ID           0
GT_weight           0
LG_weight           0
TRF_weight          0
relik_weight        0
relik_cie_weight    0
spotlight_weight    0
direction           0
dtype: int64

## Outgoing edges

In [75]:
# Read in napoleon csv files
GT_napoleon_df = pd.read_csv('../data/weight_files/napoleon_out_edges/GT_napoleon_weights.csv')

LG_napoleon_df = pd.read_csv('../data/weight_files/napoleon_out_edges/LG_napoleon_weights.csv')
TRF_napoleon_df = pd.read_csv('../data/weight_files/napoleon_out_edges/TRF_napoleon_weights.csv')
relik_napoleon_df = pd.read_csv('../data/weight_files/napoleon_out_edges/relik_napoleon_weights.csv')
relik_cie_napoleon_df = pd.read_csv('../data/weight_files/napoleon_out_edges/relik_cie_napoleon_weights.csv')
spotlight_napoleon_df = pd.read_csv('../data/weight_files/napoleon_out_edges/spotlight_napoleon_weights.csv')

TRF_napoleon_df.head()

,source_ID,target_ID,TRF_weight
0,Q517,Q7723,241
1,Q517,Q7721,241
2,Q517,Q7726,91
3,Q517,Q157521,85
4,Q517,Q151098,85


In [76]:
GT_napoleon_df = GT_napoleon_df.drop('Name', axis=1)
GT_napoleon_df = GT_napoleon_df.rename(columns={"ID": "target_ID"})
GT_napoleon_df.insert(0, 'source_ID', 'Q517')
GT_napoleon_df.head()

,source_ID,target_ID,GT_weight
0,Q517,Q15193,14
1,Q517,Q124754,13
2,Q517,Q7723,12
3,Q517,Q171480,12
4,Q517,Q7726,11


In [77]:
print(LG_napoleon_df.shape)
print(TRF_napoleon_df.shape)
print(relik_napoleon_df.shape)
print(relik_cie_napoleon_df.shape)
print(spotlight_napoleon_df.shape)
print(GT_napoleon_df.shape)

(403, 3)
(403, 3)
(403, 3)
(403, 3)
(403, 4)
(105, 3)


In [78]:
merged_napoleon = (
    LG_napoleon_df
    .merge(TRF_napoleon_df, on=['source_ID', 'target_ID'])  
    .merge(relik_napoleon_df, on=['source_ID', 'target_ID'])
    .merge(relik_cie_napoleon_df, on=['source_ID', 'target_ID'])
    .merge(spotlight_napoleon_df, on=['source_ID', 'target_ID'])    
    .merge(GT_napoleon_df,  on=['source_ID', 'target_ID'], how='left')
)
merged_napoleon['GT_weight'] = merged_napoleon['GT_weight'].fillna(0).astype(int)

merged_napoleon.head()

,source_ID,target_ID,LG_weight,TRF_weight,relik_weight,relik_cie_weight,language_code,spotlight_weight,GT_weight
0,Q517,Q7721,218,241,48,3,en,0,2
1,Q517,Q7723,216,241,2,4,en,0,12
2,Q517,Q7726,89,91,3,7,en,1,11
3,Q517,Q157521,83,85,0,0,en,0,0
4,Q517,Q151098,82,85,3,3,en,0,3


In [79]:
merged_napoleon = merged_napoleon.drop('language_code', axis=1)

In [80]:
merged_napoleon['direction'] = 'out'

In [81]:
merged_napoleon.head()

,source_ID,target_ID,LG_weight,TRF_weight,relik_weight,relik_cie_weight,spotlight_weight,GT_weight,direction
0,Q517,Q7721,218,241,48,3,0,2,out
1,Q517,Q7723,216,241,2,4,0,12,out
2,Q517,Q7726,89,91,3,7,1,11,out
3,Q517,Q157521,83,85,0,0,0,0,out
4,Q517,Q151098,82,85,3,3,0,3,out


In [82]:
merged_napoleon.shape

(403, 9)

In [83]:
merged_napoleon.isna().sum()

source_ID           0
target_ID           0
LG_weight           0
TRF_weight          0
relik_weight        0
relik_cie_weight    0
spotlight_weight    0
GT_weight           0
direction           0
dtype: int64

In [84]:
final_merged = pd.concat(
    [merged, merged_napoleon],
    ignore_index=True
)
final_merged

,source_ID,target_ID,GT_weight,LG_weight,TRF_weight,relik_weight,relik_cie_weight,spotlight_weight,direction
0,Q9068,Q517,1,1,1,0,3,1,in
1,Q360481,Q517,13,6,7,6,13,7,in
2,Q162296,Q517,2,2,2,2,5,2,in
3,Q354477,Q517,0,0,0,0,0,2,in
4,Q68771,Q517,0,0,0,0,16,3,in
...,...,...,...,...,...,...,...,...,...
444,Q517,Q151707,0,0,0,0,0,0,out
445,Q517,Q109240,0,0,0,0,0,0,out
446,Q517,Q464513,0,0,0,0,0,0,out
447,Q517,Q455181,0,0,0,0,0,0,out


In [85]:
final_merged.to_csv('../data/weight_files/all_weights.csv', index=False)